In [1]:
import pandas as pd
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import pickle

In [2]:
df=pd.read_csv('cars.csv')

In [3]:
df.head()

,Unnamed: 0,Car Name,Year,Distance,Owner,Fuel,Location,Drive,Type,Price
0,0,Maruti S PRESSO,2022.0,3878,1,PETROL,HR-98,Manual,HatchBack,514000
1,1,Hyundai Xcent,2018.0,32041,1,PETROL,TN-22,Manual,Sedan,674000
2,2,Tata Safari,2021.0,96339,1,DIESEL,TS-08,Automatic,SUV,1952000
3,3,Maruti Vitara Brezza,2019.0,51718,1,DIESEL,WB-24,Manual,SUV,690000
4,4,Tata Tiago,2021.0,19811,1,PETROL,HR-51,Manual,HatchBack,526000


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8015 entries, 0 to 8014
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  8015 non-null   int64  
 1   Car Name    8014 non-null   object 
 2   Year        8014 non-null   float64
 3   Distance    8015 non-null   int64  
 4   Owner       8015 non-null   int64  
 5   Fuel        8015 non-null   object 
 6   Location    7802 non-null   object 
 7   Drive       8015 non-null   object 
 8   Type        8015 non-null   object 
 9   Price       8015 non-null   int64  
dtypes: float64(1), int64(4), object(5)
memory usage: 626.3+ KB


In [5]:
df.isnull().sum()

Unnamed: 0      0
Car Name        1
Year            1
Distance        0
Owner           0
Fuel            0
Location      213
Drive           0
Type            0
Price           0
dtype: int64

In [6]:
df.drop("Unnamed: 0", axis=1, inplace=True)

In [7]:
df.dropna(subset=["Car Name", "Year"], inplace=True)


In [8]:
df["Location"] = df["Location"].fillna(df["Location"].mode()[0])

In [9]:
df.reset_index(drop=True, inplace=True)

In [10]:
df.isnull().sum()

Car Name    0
Year        0
Distance    0
Owner       0
Fuel        0
Location    0
Drive       0
Type        0
Price       0
dtype: int64

In [11]:
df['Year']=df['Year'].astype(int)

In [12]:
df

,Car Name,Year,Distance,Owner,Fuel,Location,Drive,Type,Price
0,Maruti S PRESSO,2022,3878,1,PETROL,HR-98,Manual,HatchBack,514000
1,Hyundai Xcent,2018,32041,1,PETROL,TN-22,Manual,Sedan,674000
2,Tata Safari,2021,96339,1,DIESEL,TS-08,Automatic,SUV,1952000
3,Maruti Vitara Brezza,2019,51718,1,DIESEL,WB-24,Manual,SUV,690000
4,Tata Tiago,2021,19811,1,PETROL,HR-51,Manual,HatchBack,526000
...,...,...,...,...,...,...,...,...,...
8009,Datsun Redi Go,2018,11500,1,PETROL,MH-01,Manual,HatchBack,292000
8010,Toyota YARIS,2018,73393,2,PETROL,KA-03,Manual,Sedan,534000
8011,Volkswagen Ameo,2016,83810,2,PETROL,UP-78,Manual,Sedan,424000
8012,Hyundai GRAND I10 NIOS,2019,39162,1,PETROL,CH-01,Automatic,HatchBack,685000


In [13]:
df['Type'].value_counts()

Type
HatchBack    5080
Sedan        1621
SUV          1171
Lux_SUV        80
Lux_sedan      62
Name: count, dtype: int64

In [14]:
numerical_cols = ["Year", "Distance", "Price"]

for col in numerical_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [15]:
df.isnull().sum()

Car Name    0
Year        0
Distance    0
Owner       0
Fuel        0
Location    0
Drive       0
Type        0
Price       0
dtype: int64

In [16]:
X=df.drop('Price',axis=1)
y=df['Price']

In [17]:
categorical_cols = [
    "Car Name",
    "Owner",
    "Fuel",
    "Location",
    "Drive",
    "Type"
]

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)


In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


In [19]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [20]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)

ridge.fit(X_train_scaled, y_train)

ridge_pred = ridge.predict(X_test_scaled)

In [21]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)

lasso.fit(X_train_scaled, y_train)

lasso_pred = lasso.predict(X_test_scaled)

c:\Users\Priya Gupta\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.249e+13, tolerance: 4.449e+10
  model = cd_fast.enet_coordinate_descent(


In [22]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

dt = DecisionTreeRegressor(random_state=42)
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

dt.fit(X_train, y_train)
rf.fit(X_train, y_train)

dt_pred = dt.predict(X_test)
rf_pred = rf.predict(X_test)

In [23]:
models = {
    'Ridge Regression': ridge,
    'Lasso Regression': lasso,
    'Decision Tree': dt,
    'Random Forest': rf
}

results = {}

for name, model in models.items():

    if name in ['Ridge Regression', 'Lasso Regression']:
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    accuracy = r2 * 100

    results[name] = accuracy

    print(
        f"{name}: "
        f"MAE = {mae:.2f}, "
        f"RMSE = {rmse:.2f}, "
        f"Accuracy = {accuracy:.2f}%"
    )

Ridge Regression: MAE = 75792.35, RMSE = 130763.56, Accuracy = 76.81%
Lasso Regression: MAE = 76059.28, RMSE = 134974.82, Accuracy = 75.29%
Decision Tree: MAE = 89192.86, RMSE = 147222.90, Accuracy = 70.60%


Random Forest: MAE = 72929.46, RMSE = 124077.65, Accuracy = 79.12%


In [24]:
import joblib

# Get best model name
best_model_name = max(results, key=results.get)

# Get actual trained model
best_model = models[best_model_name]

# Create model package
model_data = {
    "model": best_model,
    "scaler": scaler,
    "columns": X.columns.tolist(),
    "scaled": best_model_name in [
        "Ridge Regression",
        "Lasso Regression"
    ]
}

# Save
joblib.dump(model_data, "cars_price_model.pkl")

print("Best Model:", best_model_name)
print("Model saved successfully!")

Best Model: Random Forest
Model saved successfully!
